# Tutorial 00 (Duke Heart, PINN Demo): Physics-Informed Cardiac Motion and Stress

## Purpose

A self-contained demo of the physics-informed cardiac motion pipeline that
fetches every input it needs, so it runs with nothing more than a clone of the
repository -- none of Tutorials 16 and 17 has to be run first:

1. Download the ``Duke-Heart-4DLabelmaps`` gated labelmaps and two pretrained
   networks with ``monai_physio.DownloadData``:
   ``physicsnemo_physics_informed_motion_duke_heart``, a MeshGraphNet trained
   with a neo-Hookean strain-energy and incompressibility residual (PhysicsNeMo
   Sym), and ``physicsnemo_physics_informed_motion_duke_heart_ablation``, the
   same network trained on the same data with that physics term switched off.
   Each directory also carries the volumetric (tetrahedral) PCA shape model
   both were trained against.
2. Contour the reference frame of the held-out case and fit the volumetric
   shape model to it, the way Tutorial 16 fits every case: the result is a
   tetrahedral mesh of this patient's heart wall.
3. Predict the motion at every gated stage with both networks and carry the
   reference labelmap through each stage's deformation.
4. Score both: Dice against the frame acquired at each stage, and the number
   of tetrahedra whose Jacobian ``det(F)`` is not positive -- elements the
   predicted motion turns inside out.
5. Derive the Cauchy stress the physics-informed prediction implies, from the
   same neo-Hookean law its training loss used, and write an animated USD of
   the tetrahedral mesh colored by von Mises stress.

The ablation is the only fair comparison: it isolates the physics term, where
comparing against the surface MeshGraphNet of
``tutorial_00_infer_cardiac_motion_duke_heart_4d_labelmaps.ipynb`` would also
change the shape model. The physics term is a prior, so success is matching
the ablation's Dice with fewer inverted elements, not beating it.

## Hardware Required

A CUDA GPU, as for Tutorials 16 to 18.

## Data Required

Nothing -- the labelmaps, both pretrained networks and the shape model they
were trained against are all downloaded by this notebook.

## Outputs (under ``output/tutorial_00_infer_physics_informed_motion_duke_heart_4d_labelmaps/``)

  * ``pm0027_heart_surface.vtp`` -- contoured reference-frame heart
  * ``pm0027_ssm.json`` -- this patient's shape parameters
  * ``pm0027_ssm_model.vtu`` -- the volumetric model fitted to the reference frame
  * ``physics_informed/``, ``ablation/`` -- each network's
    ``pm0027_ssm_s{TTT}_pred.vtu`` predicted mesh and
    ``pm0027_ssm_s{TTT}_warped.mha`` carried labelmap per stage
  * ``pm0027_scores.csv`` -- Dice and inverted elements per stage, both networks
  * ``pm0027_physics_informed_motion.usd`` -- animated, colored by von Mises stress

In [ ]:
# Imports
import csv
import json
import logging
from typing import cast

import itk
import numpy as np
import pyvista as pv
import torch
from parameters_duke_heart_4d_labelmaps_physics import DUKE_HEART_4D_LABELMAPS_PHYSICS

from monai_physio import (
    ConvertVTKToUSD,
    DownloadData,
    EvaluateMovementDukeHeart,
    InferPhysicsNeMoMGN,
    ProcessVolumetricMesh,
    WorkflowFitStatisticalModelToPatient,
    WorkflowInferMovement,
    WorkflowInferPhysicsNeMo,
)
from monai_physio.train_physicsnemo_physics_informed_motion import (
    NeoHookeanResidual,
    compute_deformation_gradient,
    tet_edges,
    tet_volumes,
)

In [ ]:
# Data directory specification
test_mode = False

parameters = DUKE_HEART_4D_LABELMAPS_PHYSICS
case_id = parameters.hold_out_case

data_dir = parameters.hold_out_directory(test_mode)
weights_dir = parameters.weights_directory(test_mode)
output_dir = (
    parameters.output_directory(test_mode)
    / "tutorial_00_infer_physics_informed_motion_duke_heart_4d_labelmaps"
)

# The two networks Tutorial 17 trains, with the physics term and without it;
# this demo downloads pretrained copies instead. Both were trained against the
# same volumetric shape model, which ships in each directory.
model_dirs = {
    "physics_informed": parameters.physics_informed_weights_directory(test_mode),
    "ablation": parameters.ablation_weights_directory(test_mode),
}
epoch = None  # None uses the final weights
shape_model_dir = model_dirs["physics_informed"]
pca_model_file = shape_model_dir / "pca_model.json"
pca_mean_volume_file = shape_model_dir / "pca_mean.vtu"
# The tetrahedral template the networks predict on; its cells are the elements
# the strain energy was summed over.
template_file = shape_model_dir / "pca_mean_template.vtu"

number_of_pca_components = parameters.pca_components(test_mode)

# Constitutive law the physics-informed network was trained under, used again
# here to turn its predicted deformation into stress.
mu_kpa = parameters.mu_kpa
lambda_lame_kpa = parameters.lambda_lame_kpa

# Gaussian sigma, in mm, that spreads the predicted displacements into the
# continuous field the labelmap is resampled through.
smoothing_sigma_mm = 10.0

# von Mises stress is colored on log10 over this range, in kPa, so the low end
# is resolved and a few hot spots do not dominate the scale.
stress_range_kpa = (1.0, 100.0)
# Passes of neighbor averaging that remove element-to-element noise from the
# nodal stress.
stress_smoothing_iterations = 20

log_level = logging.INFO
logging.basicConfig(level=log_level)
logger = logging.getLogger(
    "tutorial_00_infer_physics_informed_motion_duke_heart_4d_labelmaps"
)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device is visible; this notebook assumes a GPU.")

output_dir.mkdir(parents=True, exist_ok=True)

## Step 1: Download the labelmaps and both pretrained networks

In [ ]:
DownloadData.DownloadDukeHeart4DLabelmapsData(data_dir)
DownloadData.DownloadPhysicsNeMoPhysicsInformedMotionDukeHeartData(weights_dir)
DownloadData.DownloadPhysicsNeMoPhysicsInformedMotionDukeHeartAblationData(weights_dir)
# One fit serves both networks only because they share one shape model.
parameters.require_matching_shape_model(model_dirs["ablation"], pca_model_file)

pca_mean_volume = cast(pv.DataSet, pv.read(str(pca_mean_volume_file)))
with pca_model_file.open(encoding="utf-8") as f:
    pca_model = json.load(f)
template = cast(pv.UnstructuredGrid, pv.read(str(template_file)))
tets = cast(np.ndarray, template.cells_dict[np.uint8(pv.CellType.TETRA)])

## Step 2: Read the gated frames and contour the reference frame

Each frame's ``g{PPP}`` gate tag gives its cardiac stage. The frame tagged
``_ref`` is the one the networks' displacements start from.

In [ ]:
frame_files = sorted((data_dir / case_id).glob("*_labelmap.nii.gz"))
reference_file = next(
    path for path in frame_files if path.name.endswith("_ref_labelmap.nii.gz")
)
cohort = EvaluateMovementDukeHeart(log_level=log_level)
stages = [cohort.stage_from_filename(path) for path in frame_files]
reference_labelmap = itk.imread(str(reference_file))
logger.info("Case %s: %d gated frames", case_id, len(frame_files))

heart_surface = parameters.whole_heart_surface(reference_file, test_mode, log_level)
heart_surface.save(str(output_dir / f"{case_id}_heart_surface.vtp"))

## Step 3: Fit the volumetric shape model to the reference frame

A volumetric template's interior nodes have no image evidence, so the fit
scores against a distance map built from the labelmap that leaves them
unpenalized. The fit warps the template with no cell-quality constraint and
can flip a handful of elements, so those are repaired before the mesh is used
as a set of physics elements.

In [ ]:
fit_workflow = WorkflowFitStatisticalModelToPatient(
    template_model=pca_mean_volume,
    patient_models=[heart_surface],
    patient_image=None,
    patient_labelmap=reference_labelmap,
    labelmap_interior_object_ids=parameters.interior_object_ids,
    log_level=log_level,
)
fit_workflow.set_use_pca_registration(
    use_pca_registration=True,
    pca_model=pca_model,
    number_of_pca_components=number_of_pca_components,
    use_surface=False,
)
fit_workflow.set_icp_transform_type(parameters.icp_transform_type)
fit_workflow.set_mask_dilation_mm(parameters.mask_dilation_mm)
fit_workflow.set_distancemap_squared_max(parameters.distancemap_squared_max)
fit_result = fit_workflow.process()

pca_coefficients = fit_workflow.pca_coefficients
assert pca_coefficients is not None
# process_time_series names every predicted mesh after this file's stem.
pca_coefficients_file = output_dir / f"{case_id}_ssm.json"
with pca_coefficients_file.open(mode="w", encoding="utf-8") as f:
    json.dump(pca_coefficients.tolist(), f)

fitted_reference_model = ProcessVolumetricMesh(
    log_level=log_level
).repair_inverted_tetrahedra(
    cast(pv.UnstructuredGrid, fit_result["fitted_reference_model"])
)
fitted_reference_model_file = output_dir / f"{case_id}_ssm_model.vtu"
fitted_reference_model.save(str(fitted_reference_model_file))
logger.info("Fitted the volumetric heart model to %s", reference_file.name)

## Step 4: Predict cardiac motion with both networks

In [ ]:
predictions = {}
for name, model_dir in model_dirs.items():
    infer_workflow = WorkflowInferPhysicsNeMo(
        model_directory=model_dir,
        inference_method=InferPhysicsNeMoMGN(
            model_tag="physics_informed_motion", log_level=log_level
        ),
        epoch=epoch,
        log_level=log_level,
    )
    predictions[name] = WorkflowInferMovement(
        infer_workflow, log_level=log_level
    ).process_time_series(
        shape_parameters=pca_coefficients_file,
        stages=stages,
        output_directory=output_dir / name,
        fitted_reference_mesh=fitted_reference_model_file,
        reference_image=reference_labelmap,
        warp_interpolation="nearest",
        warp_background_value=0.0,
        smoothing_sigma_mm=smoothing_sigma_mm,
    )

## Step 5: Score both networks

Dice is computed as in the MeshGraphNet demo: each carried labelmap against the
frame acquired at its stage, per label, averaged over labels, beside the
*no-motion* score of the reference frame itself. An element is inverted when
the Jacobian ``det(F)`` of its predicted deformation is at or below zero: the
motion has turned it inside out. The count is out of every tetrahedron in the
mesh, a few hundred thousand.

In [ ]:
def mean_label_dice(predicted: np.ndarray, acquired: np.ndarray) -> float:
    """Return the Dice of each label in ``acquired``, averaged over labels."""
    scores = []
    for label in np.unique(acquired):
        if label == 0:
            continue
        predicted_mask = predicted == label
        acquired_mask = acquired == label
        overlap = np.logical_and(predicted_mask, acquired_mask).sum()
        scores.append(2.0 * overlap / (predicted_mask.sum() + acquired_mask.sum()))
    return float(np.mean(scores))


reference_points = torch.tensor(
    np.asarray(fitted_reference_model.points), dtype=torch.float64
)
element_tets = torch.tensor(tets, dtype=torch.int64)


def deformation_gradients(predicted_mesh: pv.DataSet) -> torch.Tensor:
    """Return the per-tetrahedron deformation gradient of a predicted mesh."""
    displacement = (
        torch.tensor(np.asarray(predicted_mesh.points), dtype=torch.float64)
        - reference_points
    )
    return compute_deformation_gradient(reference_points, displacement, element_tets)


reference_array = itk.array_from_image(reference_labelmap)
rows = []
for index, (stage, frame_file) in enumerate(zip(stages, frame_files)):
    acquired = itk.array_from_image(itk.imread(str(frame_file)))
    row = {
        "stage": stage,
        "dice_no_motion": mean_label_dice(reference_array, acquired),
    }
    for name, prediction in predictions.items():
        predicted = itk.array_from_image(
            itk.imread(str(prediction["warped_images"][index]))
        )
        row[f"dice_{name}"] = mean_label_dice(predicted, acquired)
        jacobians = torch.linalg.det(
            deformation_gradients(prediction["stage_meshes"][index])
        )
        row[f"inverted_{name}"] = int((jacobians <= 0.0).sum())
    rows.append(row)

scores_file = output_dir / f"{case_id}_scores.csv"
with scores_file.open(mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print(
    f"{'stage':>6} {'Dice PINN':>10} {'Dice abl.':>10} {'no motion':>10}"
    f" {'inv. PINN':>10} {'inv. abl.':>10}"
)
for row in rows:
    print(
        f"{row['stage']:6.2f} {row['dice_physics_informed']:10.3f}"
        f" {row['dice_ablation']:10.3f} {row['dice_no_motion']:10.3f}"
        f" {row['inverted_physics_informed']:10d}"
        f" {row['inverted_ablation']:10d}"
    )
print(f"of {len(tets)} tetrahedra")

## Step 6: The stress the physics-informed motion implies

The deformation gradient is constant within a tetrahedron, so the Cauchy
stress is computed per element, averaged onto the nodes weighted by element
volume, and smoothed over the mesh edges. ``ConvertVTKToUSD`` reduces that
tensor to von Mises stress and writes the animated tetrahedral mesh colored by
it.

In [ ]:
law = NeoHookeanResidual(mu_kpa, lambda_lame_kpa, log_level=log_level)
n_points = fitted_reference_model.n_points
element_volumes, _ = tet_volumes(np.asarray(fitted_reference_model.points), tets)
corner_weights = np.repeat(element_volumes, 4)
nodal_weights = np.zeros(n_points)
np.add.at(nodal_weights, tets.ravel(), corner_weights)
edges = tet_edges(tets)
degree = np.maximum(np.bincount(edges.ravel(), minlength=n_points), 1)


def nodal_stress(predicted_mesh: pv.DataSet) -> np.ndarray:
    """Return the smoothed ``(n_points, 9)`` Cauchy stress, in kPa."""
    element_stress = (
        law.cauchy_stress(deformation_gradients(predicted_mesh)).reshape(-1, 9).numpy()
    )
    stress = np.zeros((n_points, 9))
    np.add.at(
        stress,
        tets.ravel(),
        np.repeat(element_stress, 4, axis=0) * corner_weights[:, None],
    )
    stress /= nodal_weights[:, None]
    for _ in range(stress_smoothing_iterations):
        neighbor_sum = np.zeros_like(stress)
        np.add.at(neighbor_sum, edges[:, 0], stress[edges[:, 1]])
        np.add.at(neighbor_sum, edges[:, 1], stress[edges[:, 0]])
        stress = 0.5 * stress + 0.5 * neighbor_sum / degree[:, None]
    return stress


stress_meshes = predictions["physics_informed"]["stage_meshes"]
for mesh in stress_meshes:
    mesh.point_data["stress"] = nodal_stress(mesh)

usd_file = output_dir / f"{case_id}_physics_informed_motion.usd"
converter = ConvertVTKToUSD(
    f"{case_id}_physics_informed_motion",
    stress_meshes,
    frames_per_second=float(len(stress_meshes)),
    log_level=log_level,
    output_tet_mesh=True,
)
converter.compute_von_mises_stress("stress")
log_stress_range = (
    float(np.log10(stress_range_kpa[0])),
    float(np.log10(stress_range_kpa[1])),
)
for mesh in stress_meshes:
    mesh.point_data["log10_von_mises_stress"] = np.log10(
        np.maximum(mesh.point_data["von_mises_stress"], stress_range_kpa[0])
    )
converter.set_colormap("log10_von_mises_stress", "inferno", log_stress_range)
converter.convert(str(usd_file))
logger.info("USD: %s", usd_file)

## Step 7: Look at the stress at the stage furthest from the reference

In [ ]:
furthest = int(np.argmin([row["dice_no_motion"] for row in rows]))

pv.set_jupyter_backend("static")

plotter = pv.Plotter(shape=(1, 2), window_size=[1000, 500])
plotter.subplot(0, 0)
plotter.add_mesh(fitted_reference_model.extract_surface(), color="steelblue")
plotter.add_text("Fitted reference model", font_size=10)
plotter.camera_position = "iso"

plotter.subplot(0, 1)
plotter.add_mesh(
    stress_meshes[furthest].extract_surface(),
    scalars="log10_von_mises_stress",
    cmap="inferno",
    clim=log_stress_range,
    scalar_bar_args={"title": "log10 von Mises (kPa)"},
)
plotter.add_text(f"Physics-informed, stage {stages[furthest]}", font_size=10)
plotter.camera_position = "iso"

plotter.show()